# Experiment 3.4.1 — Raw Causal Temporal Linear Readout

## Question
Does progressive fixed-duration prefix supervision improve Linear decoding of raw encoder event spike trains? Relative10 is an offline phase-normalized oracle/reference; Fixed250/Fixed500 Prefix are the causal candidates.


## Hypotheses and validation standard
Primary: `Delta250 = BA(Fixed250 Prefix) - BA(Fixed250 Whole)`. Support requires positive mean Delta250 and at least 4/5 positive paired splits. Delta500 is secondary. Future Fixed250-vs-Fixed500 selection must use mean validation BA, not test BA.


In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'scripts').is_dir() and (candidate / 'snn').is_dir():
        repo_root = candidate
        break
else:
    raise FileNotFoundError(f'Could not locate writingRing root from {cwd}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts import experiment_3_4_1_raw_causal_temporal_readout as exp341
root = exp341.results_dir(repo_root)
required = {
    'results': root / 'experiment_3_4_1_results.csv',
    'summary': root / 'experiment_3_4_1_summary.csv',
    'curves': root / 'experiment_3_4_1_prefix_curves.csv',
    'paired': root / 'experiment_3_4_1_paired_deltas.csv',
    'parity': root / 'experiment_3_4_1_baseline_parity.csv',
    'provenance': root / 'provenance.json',
}
missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Run and finalize Experiment 3.4.1 first:\n' + '\n'.join(missing))


## Artifact completeness and baseline reproduction


In [ ]:
results = pd.read_csv(required['results'])
summary = pd.read_csv(required['summary'])
curves = pd.read_csv(required['curves'])
paired = pd.read_csv(required['paired'])
parity = pd.read_csv(required['parity'])
provenance = json.loads(required['provenance'].read_text())
assert len(results) == 25
assert parity['abs_difference'].max() <= exp341.WHOLE_PARITY_TOL
display(parity)


## Final endpoint Balanced Accuracy


In [ ]:
display(summary)
fig, ax = plt.subplots(figsize=(8, 4))
labels = summary['representation'] + ' / ' + summary['supervision']
ax.errorbar(np.arange(len(summary)), summary['mean_test_ba'], yerr=summary['sd_test_ba'], fmt='o', capsize=4)
ax.set_xticks(np.arange(len(summary)), labels, rotation=30, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Final endpoint BA across five paired methods')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Paired prefix gains and oracle gaps


In [ ]:
display(paired)
delta_summary = pd.DataFrame({
    'duration': ['Fixed250', 'Fixed500'],
    'mean_delta': [paired.delta250.mean(), paired.delta500.mean()],
    'sd_delta': [paired.delta250.std(ddof=1), paired.delta500.std(ddof=1)],
    'positive_splits': [(paired.delta250 > 0).sum(), (paired.delta500 > 0).sum()],
})
display(delta_summary)
fig, ax = plt.subplots(figsize=(6, 4))
for x, col in enumerate(['delta250', 'delta500']):
    ax.scatter(np.full(len(paired), x), paired[col])
    ax.scatter([x], [paired[col].mean()], marker='D', s=80)
ax.axhline(0, linestyle='--')
ax.set_xticks([0, 1], ['Fixed250', 'Fixed500'])
ax.set_ylabel('Prefix - Whole test BA')
ax.set_title('Paired prefix supervision gain')
plt.show()


## Streaming curves
Active curves only include gestures that reach the current duration and therefore always report `n_samples` and coverage. Endpoint-aware curves carry completed gestures' endpoint predictions forward.


In [ ]:
test_active = curves[(curves.cohort == 'test') & (curves.curve_type == 'active')].copy()
agg = test_active.groupby(['representation', 'training_mode', 'time_ms'], as_index=False).agg(
    mean_ba=('balanced_accuracy', 'mean'), sd_ba=('balanced_accuracy', 'std'),
    mean_n=('n_samples', 'mean'), mean_coverage=('coverage', 'mean'))
display(agg)
for representation in ['fixed250', 'fixed500']:
    fig, ax = plt.subplots(figsize=(7, 4))
    subset = agg[agg.representation == representation]
    for mode, group in subset.groupby('training_mode', sort=False):
        ax.plot(group.time_ms, group.mean_ba, marker='o', label=mode)
    ax.set_xlabel('Observed duration (ms)')
    ax.set_ylabel('Active-gesture test BA')
    ax.set_title(f'{representation}: Whole-trained vs Prefix-trained')
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()


## Cross-duration comparison and validation-standard decision


In [ ]:
common = agg[(agg.training_mode == 'prefix') & (agg.time_ms.isin([500.0, 1000.0, 1500.0, 2000.0]))]
display(common)
primary_supported = bool((paired.delta250.mean() > 0) and ((paired.delta250 > 0).sum() >= 4))
decision = pd.DataFrame([{
    'mean_delta250': paired.delta250.mean(),
    'sd_delta250': paired.delta250.std(ddof=1),
    'positive_delta250_splits': int((paired.delta250 > 0).sum()),
    'primary_hypothesis_supported': primary_supported,
    'mean_gap250': paired.gap250.mean(),
    'mean_gap500': paired.gap500.mean(),
}])
display(decision)


## Conclusion and next experiment
If Fixed250 Prefix has positive mean paired gain and at least 4/5 positive splits, the next SNN experiment can reuse the same Fixed250 cumulative-prefix supervision. If both Fixed250 and Fixed500 Prefix fail to improve their Whole controls, supervision timing is unlikely to be the main raw-representation bottleneck and the next work should focus on SNN temporal dynamics/capacity instead.
